In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
df = pd.read_csv('final_cleaned_land.csv')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2045 entries, 0 to 2044
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   sector      2045 non-null   object 
 1   area        2045 non-null   int64  
 2   price       2045 non-null   float64
 3   open_sides  2045 non-null   int64  
 4   feature     2045 non-null   object 
dtypes: float64(1), int64(2), object(2)
memory usage: 80.0+ KB


In [6]:
df['area'].describe()

,area
count,2045.000000
mean,240.475795
std,307.988963
min,4.000000
25%,120.000000
50%,188.000000
75%,286.000000
max,4047.000000


In [ ]:
df['area'] = df['area'].astype('float')

In [ ]:
df[df['area'] > 10000][['sector','area','price']]

,sector,area,price
17,sector 4,18211.0,7.50
87,sohna road,40469.0,2.50
110,pataudi,24281.0,60.00
244,sector 113,40469.0,1.35
614,sohna road,20234.0,4.50
1416,pataudi,12141.0,3.50
1417,pataudi,16187.0,4.50
1419,sultanpur,15176.0,4.50
1420,sultanpur,16187.0,5.40
1423,pataudi,20234.0,6.50


In [ ]:
df['area'].describe(percentiles=[0.90,0.95,0.99,0.1])

,area
count,2046.000000
mean,240.358260
std,307.959544
min,0.000000
10%,67.000000
50%,188.000000
90%,420.000000
95%,508.000000
99%,926.600000
max,4047.000000


In [ ]:
df = df[df['area'] <= 4047]

In [ ]:
df = df[df['area'] > 0]

In [ ]:
df['area'].describe()

,area
count,2045.000000
mean,240.475795
std,307.988963
min,4.000000
25%,120.000000
50%,188.000000
75%,286.000000
max,4047.000000


In [ ]:
df[df['area'] > 2500]

,sector,area,price,open_sides,feature
11,sidhrawali,4047.0,5.0,1,Basic
171,sohna road,4047.0,4.0,2,Basic
234,manesar,3716.0,6.0,1,Basic
625,sohna road,4047.0,2.5,4,Standard
905,pataudi,4047.0,1.8,1,Standard
1218,sohna road,4047.0,3.0,4,Standard
1220,sohna road,4047.0,1.2,1,Standard
1421,sultanpur,4047.0,1.3,1,Basic
1424,pataudi,4047.0,4.5,1,Standard


In [ ]:
df['price'].describe(percentiles=[0.90,0.95,0.99,0.1])

,price
count,2045.000000
mean,4.443149
std,4.261586
min,0.010000
10%,0.850000
50%,3.400000
90%,9.000000
95%,11.742000
99%,21.683200
max,37.000000


In [ ]:
df[df['price'] > 22].sort_values('price',ascending=False)

,sector,area,price,open_sides,feature
1993,sector 26,836.0,37.00,2,Premium
988,sector 43,673.0,35.00,2,Premium
96,sector 25,1208.0,34.00,1,Basic
267,sector 43,809.0,32.00,2,Basic
862,sector 26,887.0,32.00,2,Premium
1641,sector 26,855.0,32.00,2,Premium
286,sector 25,1208.0,31.00,2,Basic
1658,sector 43,690.0,31.00,1,Premium
315,sector 25,910.0,30.75,3,Basic
1507,sector 26,836.0,30.00,1,Basic


In [7]:
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder()

df[['sector','feature']] = oe.fit_transform(df[['sector','feature']])

In [9]:
X = df.drop('price', axis=1)
y = df['price']

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

####1. Correlation Analysis

In [11]:
corr_imp = abs(df.corr(numeric_only=True)['price'].drop('price'))

corr_imp = corr_imp.reset_index()
corr_imp.columns = ['feature', 'corr_importance']

corr_imp

,feature,corr_importance
0,sector,0.188711
1,area,0.362327
2,open_sides,0.010548
3,feature,0.021211


####2. Random Forest Feature Importance

In [12]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X, y)

rf_imp = pd.DataFrame({
    'feature': X.columns,
    'rf_importance': rf.feature_importances_
})

rf_imp

,feature,rf_importance
0,sector,0.285557
1,area,0.681068
2,open_sides,0.014708
3,feature,0.018667


####3. Gradient Boosting Feature Importance

In [13]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    random_state=42
)

gbr.fit(X, y)

gbr_imp = pd.DataFrame({
    'feature': X.columns,
    'gbr_importance': gbr.feature_importances_
})

gbr_imp

,feature,gbr_importance
0,sector,0.317600
1,area,0.678770
2,open_sides,0.001038
3,feature,0.002592


####4. Permutation Importance

In [14]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf,
    X,
    y,
    n_repeats=10,
    random_state=42
)

perm_imp = pd.DataFrame({
    'feature': X.columns,
    'perm_importance': perm.importances_mean
})

perm_imp

,feature,perm_importance
0,sector,0.670097
1,area,1.275285
2,open_sides,0.022097
3,feature,0.036861


####5. SHAP

In [15]:
pip install shap

In [16]:
import shap

explainer = shap.TreeExplainer(rf)

shap_values = explainer.shap_values(X)

shap_imp = pd.DataFrame({
    'feature': X.columns,
    'shap_importance': np.abs(shap_values).mean(axis=0)
})

shap_imp

,feature,shap_importance
0,sector,1.324006
1,area,2.224991
2,open_sides,0.061591
3,feature,0.096352


In [17]:
final = corr_imp.merge(
    rf_imp,
    on='feature'
)

final = final.merge(
    gbr_imp,
    on='feature'
)

final = final.merge(
    perm_imp,
    on='feature'
)

final = final.merge(
    shap_imp,
    on='feature'
)

final

,feature,corr_importance,rf_importance,gbr_importance,perm_importance,shap_importance
0,sector,0.188711,0.285557,0.317600,0.670097,1.324006
1,area,0.362327,0.681068,0.678770,1.275285,2.224991
2,open_sides,0.010548,0.014708,0.001038,0.022097,0.061591
3,feature,0.021211,0.018667,0.002592,0.036861,0.096352


In [19]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

cols = [
    'corr_importance',
    'rf_importance',
    'gbr_importance',
    'perm_importance',
    'shap_importance'
]

final[cols] = scaler.fit_transform(
    final[cols]
)

In [20]:
final['avg_importance'] = final[
    cols
].mean(axis=1)

final.sort_values(
    'avg_importance',
    ascending=False,
    inplace=True
)

final

,feature,corr_importance,rf_importance,gbr_importance,perm_importance,shap_importance,avg_importance
1,area,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000
0,sector,0.506463,0.40646,0.467090,0.517081,0.583533,0.496125
3,feature,0.030311,0.00594,0.002292,0.011781,0.016068,0.013278
2,open_sides,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000


Feature importance analysis was performed using Correlation Analysis, Random Forest, Gradient Boosting, Permutation Importance and SHAP values. The average importance across all techniques showed that Area is the most influential predictor of land price, followed by Sector. Feature category has negligible impact, while Open Sides contributes almost no predictive power.

In [28]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

X_all = df[['area','sector','feature','open_sides']]
y = df['price']

scores = cross_val_score(
    rf,
    X_all,
    y,
    cv=5,
    scoring='r2'
)

print("All Features CV R2 :", scores.mean())

All Features CV R2 : 0.7974351932791902


In [27]:
X_drop = df[['area','sector','feature']]

scores = cross_val_score(
    rf,
    X_drop,
    y,
    cv=5,
    scoring='r2'
)

print("Without Open Sides CV R2 :", scores.mean())

Without Open Sides CV R2 : 0.8016740400800517


In [29]:
final_df = df.drop(columns=['open_sides'])

In [30]:
final_df.to_csv(
    'land_post_feature_selection.csv',
    index=False
)